In [18]:
import spacy
from Bio import Entrez
import pandas as pd
import re
import ssl
import os
ssl._create_default_https_context = ssl._create_unverified_context

In [10]:
# 1. Ask user to input the variant they want to check

# Variant = input("Enter variant (e.g., BRCA1, V600E): ")
variant = "BRCA1 c.5266dupC"

# Option to filter by date
#use_date= input("Filter by year? ")
date_filter = "2020"

# Build search query with date filter
query = f"{variant} AND {date_filter}[PDAT]"

print(f"Gene: {variant}")
print(f"Date filter: {date_filter}")

Gene: BRCA1 c.5266dupC
Date filter: 2020


In [33]:
#2. Download papers from PubMed 

from Bio import Entrez
from Bio import Medline

Entrez.email = "jawahir_noor@hotmail.com"


# Search
handle = Entrez.esearch(db="pubmed", term=query, retmax=10)
record = Entrez.read(handle)
ids = record["IdList"]
handle.close()

# Simpler approach using BioPython's built-in parser
# Fetch in Medline format
handle = Entrez.efetch(db="pubmed", id=ids, rettype="medline", retmode="text")
records = Medline.parse(handle)

papers_list = []
for record in records:
    papers_list.append({
        "title": record.get("TI", ""),
        "abstract": record.get("AB", ""),
        "pmid": record.get("PMID", "")
    })
handle.close()

print(papers_list)



[{'title': 'Spectrum of BRCA1/2 Mutations in Romanian Breast and Ovarian Cancer Patients.', 'abstract': 'Background: About 10,000 women are diagnosed with breast cancer and about 2000 women are diagnosed with ovarian cancer each year in Romania. There is an insufficient number of genetic studies in the Romanian population to identify patients at high risk of inherited breast and ovarian cancer. Methods: We evaluated 250 women of Romanian ethnicity with BC and 240 women of Romanian ethnicity with ovarian cancer for the presence of damaging germline mutations in breast cancer genes 1 and 2 (BRCA1 and BRCA2, respectively) using Next-Generation Sequencing (NGS) technology. Results: Of the 250 breast cancer patients, 47 carried a disease-predisposing BRCA mutation (30 patients (63.83%) with a BRCA1 mutation and 17 patients (36.17%) with a BRCA2 mutation). Of the 240 ovarian cancer patients, 60 carried a BRCA mutation (43 patients (72%) with a BRCA1 mutation and 17 patients (28%) with a BRCA

In [12]:
#3. Load model (BioNLP13CG)

# BioNLP model (biomedical entities)
nlp_bionlp = spacy.load("en_ner_bionlp13cg_md")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_ner_bionlp13cg_md' (0.5.1) was trained with spaCy v3.4.1 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [13]:
#4. Extract biomedical entities (BioNLP13CG ) paper by paper
import pandas as pd
import re

results = []


for i, paper in enumerate(papers_list):

    text = paper["abstract"]

    if not text:
        continue

    # safe limit
    text = text[:5000]

    
    doc_bio = nlp_bionlp(text)

    # BioNLP entities
    for ent in doc_bio.ents:
        results.append({
            "paper_id": i,
            "title": paper["title"],
            "text": ent.text,
            "label": ent.label_,
            "model": "BioNLP13CG"
        })

The BioNLP13CG model was used to extract biomedical entities such as genes, diseases, and organisms from the collected abstracts. However, this model does not capture structured genomic variant formats (e.g., c.6313delA or p.Arg175His), as it is not specifically trained for mutation recognition. 

To address this limitation, a rule-based approach using regular expressions is implemented to identify variant mentions based on common HGVS-like patterns. The extracted variants are then added to the results as a separate category, allowing the pipeline to combine machine learning-based entity recognition with rule-based mutation extraction for more complete genomic information retrieval.

In [19]:
 # 5. Extract mutation variants (Rule-based + regex) 
mutation_patterns = [
    r"c\.\d+(?:_\d+)?[A-Z]>[A-Z]",        # c.5266C>T  (strict: single base change)
    r"c\.\d+(?:_\d+)?dup[A-Z]?",           # c.5266dupC or c.5266dup
    r"c\.\d+(?:_\d+)?del[A-Z]*",           # c.5266del
    r"c\.\d+(?:_\d+)?ins[A-Z]+",           # c.5266insA
    r"p\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}",  # p.Gln1756Ter  (strict 3-letter AA)
    r"rs\d{4,}",                            # rs ID (min 4 digits to avoid false hits)
]

def is_valid_variant(v):
    """Reject malformed extractions like 'c1016d', single-letter hits, etc."""
    # Must start with c., p., or rs
    if not re.match(r"^(c\.|p\.|rs)", v, re.IGNORECASE):
        return False
    # c. variants must have a numeric position
    if v.startswith("c.") and not re.search(r"c\.\d+", v):
        return False
    return True
 
def extract_variants(text):
    variants = []
    for pattern in mutation_patterns:
        matches = re.findall(pattern, text)
        variants.extend(matches)
    return [v for v in set(variants) if is_valid_variant(v)]
 
variant_results = []
for i, paper in enumerate(papers_list):
    text = paper["abstract"]
    if not text:
        continue
    for variant_match in extract_variants(text[:5000]):
        variant_results.append({
            "paper_id": i,
            "title": paper["title"],
            "pmid": paper.get("pmid", ""),
            "variant": variant_match
        })

In [30]:

# 6. LitVar2 normalization

LITVAR2_BASE = "https://www.ncbi.nlm.nih.gov/research/litvar2-api"
 
def litvar2_normalize(variant_name, gene_name):
    """
    Normalize a variant using the LitVar2 API.
 
    Step 1 — autocomplete: query LitVar2 with the variant string.
              Returns a list of candidates, each with rsid, gene(s), pmids_count.
    Step 2 — pick the best candidate that matches our gene.
    Step 3 — fetch linked PMIDs for that rsid.
 
    Returns a dict with: rsid, litvar_name, gene, pmids_count, pmids (list)
    """
    try:
        # Step 1: autocomplete search
        url = f"{LITVAR2_BASE}/variant/autocomplete/?query={quote(variant_name)}"
        r = requests.get(url, timeout=15)
        time.sleep(0.34)
 
        if r.status_code != 200 or not r.json():
            return None
 
        candidates = r.json()
 
        # Step 2: pick best candidate matching our gene
        best = None
        for candidate in candidates:
            candidate_genes = candidate.get("gene", [])
            # Check if our gene appears in the candidate's gene list
            if any(gene_name.upper() == g.upper() for g in candidate_genes):
                best = candidate
                break
 
        # If no gene match, take the first result (may still be relevant)
        if best is None and candidates:
            best = candidates[0]
 
        if best is None:
            return None
 
        rsid        = best.get("rsid", "N/A")
        litvar_name = best.get("name", variant_name)
        genes       = best.get("gene", [])
        pmids_count = best.get("pmids_count", 0)
 
        # Step 3: fetch linked PMIDs for this rsid
        pmids = []
        if rsid != "N/A" and int(pmids_count) > 0:
            encoded_id = f"litvar%40{rsid}%23%23"
            pub_url = f"{LITVAR2_BASE}/variant/get/{encoded_id}/publications"
            r2 = requests.get(pub_url, timeout=15)
            time.sleep(0.34)
            if r2.status_code == 200:
                pub_data = r2.json()
                pmids = pub_data.get("pmids", [])
 
        return {
            "rsid":        rsid,
            "litvar_name": litvar_name,
            "gene":        ", ".join(genes) if genes else gene_name,
            "pmids_count": pmids_count,
            "pmids":       pmids,
        }
 
    except Exception:
        return None


In [31]:
# 6.b. Build tables, normalize, print and save

os.makedirs("results", exist_ok=True)
 
df_entities = pd.DataFrame(results)
# Define gene from your variant
gene = variant.split()[0]
 
# Normalize via LitVar2 and add columns directly into variant_results rows
print("\nNormalizing variants via LitVar2...")
for row in variant_results:
    result = litvar2_normalize(row["variant"], gene)
 
    if result:
        row["rsid"]        = result["rsid"]
        row["litvar_name"] = result["litvar_name"]
        row["pmids_count"] = result["pmids_count"]
        row["litvar_pmids"] = "|".join(str(p) for p in result["pmids"][:5])  # first 5 PMIDs
        print(f"  {row['variant']} → {result['rsid']} | {result['litvar_name']} | {result['pmids_count']} papers")
    else:
        row["rsid"]        = "N/A"
        row["litvar_name"] = "N/A"
        row["pmids_count"] = 0
        row["litvar_pmids"] = "N/A"
        print(f"  {row['variant']} → Not found in LitVar2")
 
df_variants = pd.DataFrame(variant_results)
 
# Print 
print("\n=== Entities Found ===")
print(df_entities.head())
print(f"\nTotal entities: {len(df_entities)}")
 
print("\n=== Variants Found ===")
print(df_variants[["pmid", "variant", "rsid", "litvar_name", "pmids_count", "litvar_pmids"]].head(20))
print(f"\nTotal variants: {len(df_variants)}")
 
# Save 
df_entities.to_csv("results/paper_by_paper_entities.csv", index=False)
df_variants.to_csv("results/extracted_variants.csv", index=False)
 
print("\nResults saved to 'results/' directory")



Normalizing variants via LitVar2...
  c.5266dupC → rs80357906 | c.5266dupC | 476 papers
  c.9371A>T → rs28897759 | p.N3124I | 70 papers
  c.3607C>T → rs62625308 | c.3607C>T | 100 papers
  c.3279delC → rs397509050 | c.3279delC | 14 papers
  c.181T>G → rs28897672 | p.C61G | 547 papers
  c.68_69del → rs80357914 | p.E23fsX | 140 papers
  c.798_799delTT → rs80357724 | c.798_799delTT | 80 papers
  c.5266dup → rs80357906 | c.5266dupC | 476 papers
  c.5266dupC → rs80357906 | c.5266dupC | 476 papers
  c.5309G>T → rs863224765 | c.5309G>T | 33 papers
  c.211dupA → rs397508938 | c.211dupA | 26 papers

=== Entities Found ===
   paper_id                                              title      pmid  \
0         0  Spectrum of BRCA1/2 Mutations in Romanian Brea...  35409996   
1         0  Spectrum of BRCA1/2 Mutations in Romanian Brea...  35409996   
2         0  Spectrum of BRCA1/2 Mutations in Romanian Brea...  35409996   
3         0  Spectrum of BRCA1/2 Mutations in Romanian Brea...  35409996   

In [34]:
# 7. Evaluation matrix

from sklearn.metrics import precision_score, recall_score, f1_score

# What the pipeline extracted
found = {
    0: ["c.5266dupC", "c.3607C>T", "c.9371A>T"],  # Paper 0 - Romanian study
    1: ["c.5266dupC", "c.181T>G", "c.211dupA","c.181T>G","c.68_69del","c.798_799delTT"],   # Paper 1 - North African study
}

# Gold standard (what found in each paper)
correct = {
    0: ["c.5266dupC", "c.3607C>T", "c.9371A>T", "c.181T>G","c.1687C>T", "c.68_69delAG", "c.4218delG"],
    1: ["c.211dupA", "c.798_799delTT", "c.5266dupC", "c.5309G>T","c.3279delC", "c.1310_1313delAAGA", "c.68_69delAG", "c.181T>G"]
}

y_true = [1, 1]  # papers actually have it 1=has mutation, 0=no mutation
y_pred = [1, 1]  # the model detected

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Precision: {precision:.1%}")
print(f"Recall: {recall:.1%}")
print(f"F1: {f1:.1%}")

Precision: 100.0%
Recall: 100.0%
F1: 100.0%
